# 01 — Supervised fine-tuning (SFT) on CPU, end to end

This lab walks through the full SageMaker fine-tuning workflow, the same five steps as the GPU labs, on CPU
instances only. It runs in any account without waiting for GPU capacity, in about 25 minutes.

| Step | What happens | What it shows |
|---|---|---|
| 1. Prepare | generate 400 training / 30 test examples, write them as JSONL, upload them to S3 | how a job receives your data (**input channels**) |
| 2. Fine-tune | a training job trains a LoRA adapter on **Qwen3-0.6B** (20 steps), launched 3 times in parallel: **on-demand**, **Spot**, and an instance-type **fallback** list | the anatomy of a training job; how to get compute when a type is scarce, and what Spot saves |
| 3. Evaluate | a separate **evaluation job** scores the base and the fine-tuned model on the test data; a **quality gate** registers the model in the **Model Registry** as approved or rejected; metrics go to **MLflow** | measuring what fine-tuning changed, and only promoting a model that passed |
| 4. Deploy | deploy the approved model 3 ways: **real-time**, real-time with **instance pools** (fallback), **serverless**; compare base and fine-tuned answers | the deployment options and their trade-offs |
| 5. Clean up | delete the endpoints (and optionally the registry entries and S3 data) | what keeps costing money |

**The task.** Turn a free-text restaurant booking request into strict JSON:

`"hey! Dana - four ppl friday 7pm if possible"` → `{"name": "Dana", "party_size": 4, "day": "Friday", "time": "19:00"}`

The base model gets the same instruction and usually gets the values roughly right, but not the exact format:
it wraps the JSON in a markdown code block, keeps "friday" lowercase, writes "7pm". The metric is **exact JSON
match**, so it starts near 0%; a few LoRA steps teach the format.

**Where things end up** (all under your SageMaker default bucket, prefix `cpu-fine-tuning-techniques/`):

```
data/train/train.jsonl, data/test/test.jsonl          step 1: the dataset
jobs/<training-job>/output/model/adapter/             step 2: the LoRA adapter (~9 MB): weights, config, tokenizer
jobs/<training-job>/output/model/training_metrics.json step 2: loss per step, trainable parameters, timings
jobs/<training-job>/output/model/serving/model.tar.gz step 3: adapter + inference code, registered and deployed
jobs/<evaluation-job>/output/model/eval_results.json  step 3: base vs fine-tuned scores, example outputs
```

**Before you start.** Check these quotas in the Service Quotas console (service *Amazon SageMaker*):
`ml.c7i.4xlarge for training job usage` ≥ 2 (on-demand + evaluation), `ml.m5.4xlarge for spot training job usage`
≥ 1 (the Spot job; without it only that job fails), `ml.m5.xlarge for endpoint usage` ≥ 2 (the real-time endpoint
and the first instance pool; the pools also try `ml.m7i.xlarge` and `ml.c7i.2xlarge`), and serverless endpoint
memory. Cost is small: four ~4-minute CPU jobs plus endpoints for about 10 minutes, deleted in step 5.

The model is small and the run short on purpose: this lab shows the mechanics. The same code pattern scales to
GPU instances and larger models; notebooks 02-04 apply it to other techniques (continued pre-training, DPO,
GRPO).

In [ ]:
%pip install -q "sagemaker==3.21.0"

## Setup

The SageMaker session, execution role and default S3 bucket, and MLflow. `MLFLOW_APP_ARN = ""` reuses the
account's MLflow App or creates one (several minutes, first time only); set an ARN to use a specific app, or
`None` to skip MLflow. The jobs run the same either way.

In [ ]:
import json
import time

import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

from sm_helpers import mlflow_environment, mlflow_url, resolve_mlflow

sess = Session()
region = sess.boto_region_name
try:
    role = get_execution_role()
except ValueError:
    role = boto3.client("iam").get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]
bucket = sess.default_bucket()
prefix = "/".join(p for p in (sess.default_bucket_prefix, "cpu-fine-tuning-techniques") if p)
sm = boto3.client("sagemaker", region_name=region)
s3 = boto3.client("s3", region_name=region)

MLFLOW_APP_ARN = ""
mlflow_arn = resolve_mlflow(sm, sess, MLFLOW_APP_ARN)

data_uri = {split: f"s3://{bucket}/{prefix}/data/{split}/" for split in ("train", "test")}
MODEL_PACKAGE_GROUP = "cpu-ft-sft-booking"
print(f"role:   {role}\nregion: {region}\noutput: s3://{bucket}/{prefix}/\nMLflow: {mlflow_arn or 'off'}")


def read_json(s3_uri):
    b, k = s3_uri[5:].split("/", 1)
    return json.loads(s3.get_object(Bucket=b, Key=k)["Body"].read())


def lab_endpoints():
    """The newest endpoint of each kind created in step 4 ({kind: name}): used after a kernel restart."""
    found = {}
    for e in sm.list_endpoints(NameContains="cpu-ft-sft-", SortBy="CreationTime", SortOrder="Ascending",
                               MaxResults=100)["Endpoints"]:
        if e["EndpointStatus"] != "Deleting":
            found[e["EndpointName"][len("cpu-ft-sft-"):].rsplit("-", 2)[0]] = e["EndpointName"]
    return found


def wait_for(job_name, poll=15):
    """Prints status changes until the job ends; safe to interrupt and re-run."""
    last = None
    while True:
        d = sm.describe_training_job(TrainingJobName=job_name)
        line = f"{d['TrainingJobStatus']}/{d['SecondaryStatus']}"
        if line != last:
            print(time.strftime("%H:%M:%S"), job_name, line, flush=True)
            last = line
        if d["TrainingJobStatus"] in ("Completed", "Failed", "Stopped"):
            return d
        time.sleep(poll)

## 1. Prepare the data

Generate the examples (`scripts/booking_data.py`), write them as JSONL, one `{"request": ..., "target": ...}` per
line, and upload them to S3. The jobs get them as **input channels**: SageMaker copies each S3 prefix to
`/opt/ml/input/data/<channel>/` before the script starts. The training job gets `train`; the evaluation job gets
`test`, which training never sees. To use your own data, upload files in the same format and point the channels at
them.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "scripts")
from booking_data import INSTRUCTION, make_splits

train, test = make_splits(n_train=400, n_test=30)
print(f"instruction: {INSTRUCTION}\n{len(train)} training / {len(test)} test examples\n")
for ex in train[:3]:
    print(f"{ex['request']}\n  -> {ex['target']}")
print()
for split, rows in (("train", train), ("test", test)):
    local = Path("data") / split / f"{split}.jsonl"
    local.parent.mkdir(parents=True, exist_ok=True)
    local.write_text("".join(json.dumps(r) + "\n" for r in rows))
    s3.upload_file(str(local), bucket, f"{prefix}/data/{split}/{local.name}")
    print(f"{split:5} -> {data_uri[split]}{local.name}")

## 2. Fine-tune: three ways to get capacity

The training job (`scripts/train.py`) reads the `train` channel, trains the LoRA adapter for 20 steps and writes
`adapter/` and `training_metrics.json` to `/opt/ml/model`, which SageMaker uploads to S3 when the job ends. It
does not evaluate: that is step 3, in a separate job. The same job is launched three times, in parallel, each
asking SageMaker for capacity in a different way:

| `capacity` | What SageMaker does | When to use it |
|---|---|---|
| `on-demand` | runs on exactly `INSTANCE_TYPE` | the default; predictable |
| `spot` | runs on `SPOT_INSTANCE_TYPE` using spare (Spot) capacity, waiting up to `max_wait` for it; can be interrupted | cheaper; for jobs that can wait, or resume from checkpoints |
| `fallback` | runs on the first type in `FALLBACK_TYPES` that has capacity (`instance_preferences`) | when one type is often unavailable (typical for GPUs) |

SageMaker rejects a fallback list if any type in it has no training quota, so types without quota are dropped
first. Outputs are stored uncompressed so the next steps can use the adapter directly.

**Picking a Spot type.** Spot capacity varies by type and over time, so the Spot job uses its own type,
`SPOT_INSTANCE_TYPE` (`ml.m5.4xlarge`: widely available, same 16 vCPUs). To compare types before choosing, the
EC2 **Spot placement score** rates the chance of getting Spot capacity (1-10, per Region or Availability Zone):
EC2 console > **Spot Requests** > **Spot placement score**, or
`aws ec2 get-spot-placement-scores --instance-types m5.4xlarge --target-capacity 1 --region-names us-east-1 --single-availability-zone`.
It describes EC2 Spot capacity, which SageMaker training draws on, so treat it as a strong hint, not a guarantee.

In [ ]:
from botocore.config import Config
from sagemaker.core.shapes import InstancePreference
from sagemaker.core.training.configs import Compute, InputData, OutputDataConfig, SourceCode, StoppingCondition
from sagemaker.train import ModelTrainer

STEPS = 20
INSTANCE_TYPE = "ml.c7i.4xlarge"
SPOT_INSTANCE_TYPE = "ml.m5.4xlarge"  # Spot capacity differs by type; see "Picking a Spot type" above
FALLBACK_TYPES = ["ml.c7i.4xlarge", "ml.m7i.4xlarge", "ml.c6i.4xlarge", "ml.m6i.4xlarge", "ml.m5.4xlarge"]

try:
    # adaptive retries: notebooks started together can hit the Service Quotas rate limit
    sq = boto3.client("service-quotas", region_name=region, config=Config(retries={"max_attempts": 10, "mode": "adaptive"}))
    # applied values override defaults; a quota still at its default is only in the defaults list
    quotas = {}
    for op in ("list_aws_default_service_quotas", "list_service_quotas"):
        for page in sq.get_paginator(op).paginate(ServiceCode="sagemaker"):
            quotas.update({q["QuotaName"]: q["Value"] for q in page["Quotas"]})
    FALLBACK_TYPES = [t for t in FALLBACK_TYPES if quotas.get(f"{t} for training job usage", 0) >= 1] or FALLBACK_TYPES
except Exception as e:  # role cannot read Service Quotas: send the full list
    print(f"quota check skipped: {e}")
print(f"fallback list: {FALLBACK_TYPES}")


def make_trainer(capacity, entry_script, hyperparameters, base_job_name):
    if capacity == "fallback":
        compute = Compute(instance_preferences=[InstancePreference(instance_type=t) for t in FALLBACK_TYPES],
                          instance_count=1, volume_size_in_gb=30)
    else:
        compute = Compute(instance_type=SPOT_INSTANCE_TYPE if capacity == "spot" else INSTANCE_TYPE,
                          instance_count=1, volume_size_in_gb=30, enable_managed_spot_training=capacity == "spot")
    return ModelTrainer(
        training_image=f"763104351884.dkr.ecr.{region}.amazonaws.com/pytorch-training:2.8-cpu-py312",
        source_code=SourceCode(source_dir="scripts", requirements="requirements.txt", entry_script=entry_script),
        hyperparameters=hyperparameters,
        environment=mlflow_environment(mlflow_arn),  # the job logs to MLflow when an app is set
        compute=compute,
        # Spot only: max_wait is the total time allowed, including waiting for Spot capacity; past it the job
        # stops instead of waiting for capacity indefinitely. Tighter than in notebooks 02-04 (sm_helpers.Lab) because
        # these jobs are short and the notebook waits for them
        stopping_condition=StoppingCondition(max_runtime_in_seconds=1200,
                                             max_wait_time_in_seconds=1800 if capacity == "spot" else None),
        output_data_config=OutputDataConfig(s3_output_path=f"s3://{bucket}/{prefix}/jobs", compression_type="NONE"),
        base_job_name=base_job_name,
        role=role,
        sagemaker_session=sess,
    )

Run the definitions above again after a kernel restart; the next cell starts the three jobs (run it once).

In [ ]:
jobs = {}
for capacity in ("on-demand", "spot", "fallback"):
    trainer = make_trainer(capacity, "train.py", {"steps": STEPS, "model_id": "Qwen/Qwen3-0.6B"},
                           f"cpu-ft-sft-{capacity}")
    # The S3 prefix from step 1 becomes /opt/ml/input/data/train in the job
    trainer.train(input_data_config=[InputData(channel_name="train", data_source=data_uri["train"])], wait=False)
    jobs[capacity] = trainer._latest_training_job.training_job_name
    print(f"{capacity:10} submitted: {jobs[capacity]}")

### Wait for the three jobs

The jobs run in SageMaker independently of this notebook. If the kernel restarts, re-run the setup cell and this
one: it finds the latest job of each kind. Once on-demand has finished, the cell waits at most 10 more minutes
for the others (Spot can wait a long time for capacity), then continues.

In [ ]:
jobs = globals().get("jobs") or {
    c: sm.list_training_jobs(NameContains=f"cpu-ft-sft-{c}", SortBy="CreationTime", SortOrder="Descending",
                             MaxResults=1)["TrainingJobSummaries"][0]["TrainingJobName"]
    for c in ("on-demand", "spot", "fallback")}
GRACE_S = 600  # once on-demand is done, wait at most this long for the others (e.g. Spot without capacity)
last, done_at = {}, None
while True:
    desc = {c: sm.describe_training_job(TrainingJobName=n) for c, n in jobs.items()}
    for c, d in desc.items():
        line = f"{d['TrainingJobStatus']}/{d['SecondaryStatus']}"
        if last.get(c) != line:
            print(time.strftime("%H:%M:%S"), f"{c:10}", line, flush=True)
            last[c] = line
    finished = {c for c, d in desc.items() if d["TrainingJobStatus"] in ("Completed", "Failed", "Stopped")}
    if len(finished) == len(desc):
        break
    if "on-demand" in finished:
        done_at = done_at or time.time()
        if time.time() - done_at > GRACE_S:
            print(f"\nstill running: {sorted(set(desc) - finished)}. Continuing with the on-demand model; "
                  "re-run this cell later to see how they ended.")
            break
    time.sleep(15)


def phases(d):
    out = {}
    for t in d.get("SecondaryStatusTransitions", []):
        end = t.get("EndTime") or d.get("TrainingEndTime")
        if end:
            out[t["Status"]] = (end - t["StartTime"]).total_seconds()
    return out


print(f"\n{'capacity':10} {'status':10} {'instance':16} {'to start':>8} {'training':>8} {'billed':>7} {'spot savings':>12}")
for c, d in desc.items():
    rc, p = d["ResourceConfig"], phases(d)
    savings = ""
    if c == "spot" and d.get("TrainingTimeInSeconds"):
        savings = f"{1 - d['BillableTimeInSeconds'] / d['TrainingTimeInSeconds']:.0%}"
    print(f"{c:10} {d['TrainingJobStatus']:10} {rc.get('SelectedInstanceType') or rc.get('InstanceType') or '-':16} "
          f"{p.get('Starting', 0) + p.get('Downloading', 0):7.0f}s {p.get('Training', 0):7.0f}s "
          f"{d.get('BillableTimeInSeconds', 0):6}s {savings:>12}")

**How to read the table.** One row per capacity option:

| Column | Meaning |
|---|---|
| status | `Completed`, or `Failed`/`Stopped` (e.g. Spot found no capacity within `max_wait`: `MaxWaitTimeExceeded`) |
| instance | the instance the job ran on; for `fallback`, the type SageMaker picked from the list |
| to start | time from submission until the script started: waiting for capacity, starting the instance, downloading the container and data |
| training | time the script ran (install requirements, download the model, train, save) |
| billed | seconds you pay for. Spot bills the same run time at the Spot discount, shown as "spot savings" (`1 - billable time / training time`) |

The three jobs train the same model on the same data; only the capacity path differs. Most of the time is
startup, not training: that is the fixed cost of a job, and why short experiments are often run on a dev
instance first.

### What the training job produced

In [ ]:
d = sm.describe_training_job(TrainingJobName=jobs["on-demand"])
model_prefix = d["ModelArtifacts"]["S3ModelArtifacts"].rstrip("/") + "/"
adapter_uri = model_prefix + "adapter/"
tm = read_json(model_prefix + "training_metrics.json")
print(f"adapter:   {adapter_uri}\n           https://s3.console.aws.amazon.com/s3/buckets/{bucket}?prefix={adapter_uri[6 + len(bucket):]}")
print(f"trained:   {tm['trainable_parameters'] / 1e6:.1f} M parameters (the LoRA matrices; the 0.6 B base weights are frozen)")
print(f"loss:      {tm['loss'][0]:.3f} (step 1) -> {tm['loss'][-1]:.3f} (step {tm['steps']}), {tm['seconds_training']} s of training")
for obj in s3.list_objects_v2(Bucket=bucket, Prefix=adapter_uri[6 + len(bucket):])["Contents"]:
    print(f"   {obj['Size'] / 1e6:7.2f} MB  {obj['Key'].rsplit('/', 1)[-1]}")

The training loss is the error on the training examples' answers; it falls quickly because the format is easy to
learn. It does not tell you how the model does on requests it has not seen, or whether it beats the base model:
that is what the evaluation measures. The adapter is the only thing training changed: `adapter_model.safetensors`
holds the LoRA weights, `adapter_config.json` names the base model they apply to.

## 3. Evaluate, then register

A separate **evaluation job** (`scripts/evaluate.py`) gets two channels, the adapter from step 2 and the `test`
data from step 1. It loads the base model named in the adapter's config, scores it on the 30 test requests, adds
the adapter, scores again on the same requests, and writes `eval_results.json`.

Why a separate job rather than evaluating at the end of training: the evaluation is the same for every candidate
(any adapter, from any training run, can be scored the same way), it runs on data training never had access to,
and its result is a record of its own that the next step can rely on.

In [ ]:
evaluator = make_trainer("fallback", "evaluate.py", {"adapter_s3_uri": adapter_uri}, "cpu-ft-sft-eval")
evaluator.train(input_data_config=[InputData(channel_name="adapter", data_source=adapter_uri),
                                   InputData(channel_name="test", data_source=data_uri["test"])], wait=False)
eval_job = evaluator._latest_training_job.training_job_name
print("submitted:", eval_job)

In [ ]:
eval_job = globals().get("eval_job") or sm.list_training_jobs(
    NameContains="cpu-ft-sft-eval", SortBy="CreationTime", SortOrder="Descending", MaxResults=1)["TrainingJobSummaries"][0]["TrainingJobName"]
d = wait_for(eval_job)
eval_uri = d["ModelArtifacts"]["S3ModelArtifacts"].rstrip("/") + "/eval_results.json"
results = read_json(eval_uri)
print(f"\n{'':10} {'exact':>6} " + " ".join(f"{f:>10}" for f in ("name", "party_size", "day", "time")))
for which in ("base", "finetuned"):
    m = results[which]
    print(f"{which:10} {m['exact_match']:6.0%} " + " ".join(f"{m['acc_' + f]:10.0%}" for f in ("name", "party_size", "day", "time")))
print()
for ex in results["examples"][:2]:
    print(f"request:   {ex['request']}\nbase:      {ex['base']!r}\nfinetuned: {ex['finetuned']}\n")

### How to read the results

| Row | Model |
|---|---|
| `base` | Qwen3-0.6B as downloaded, given the same instruction |
| `finetuned` | the same model + the LoRA adapter from step 2 |

| Column | What it measures (on the 30 test requests, never used in training) |
|---|---|
| exact | the answer parses as JSON and equals the target exactly: bare JSON (no markdown code block), `"Friday"` capitalized, 24-hour `"19:00"` |
| name, party_size, day, time | per field: the share of answers where that field is right. The base model often has the values right inside a code block or in another format, which counts as wrong here |

The example requests show the difference: the base model's answer is close but not usable by a program
expecting strict JSON; the fine-tuned answer is.

### Quality gate and Model Registry

The **SageMaker Model Registry** keeps versioned model packages: what to deploy (container image, model data in
S3), with an approval status and metadata. Here each evaluated adapter becomes a version in the package group
`cpu-ft-sft-booking`, with its evaluation attached:

* **Approved** if it passes the gate: fine-tuned exact match at least `MIN_EXACT_MATCH` and better than the base
  model. Step 4 deploys only approved versions.
* **Rejected** otherwise. It stays in the registry with its metrics, so you can see why.

The model data is `model.tar.gz`: the adapter plus the inference code (`serving/inference.py`). The base model is
not in it; the endpoint downloads it from the Hugging Face Hub at startup.

In [ ]:
import tarfile
import tempfile

MIN_EXACT_MATCH = 0.8

# Register exactly the adapter the evaluation scored (recorded by evaluate.py), also after a kernel restart
adapter_uri = results["adapter"]
model_prefix = adapter_uri.rsplit("adapter/", 1)[0]
training_job = adapter_uri.split("/jobs/", 1)[1].split("/", 1)[0]

# Package the adapter (~9 MB) + inference code as model.tar.gz (serverless endpoints need a tarball)
work = Path(tempfile.mkdtemp())
adapter_key = adapter_uri[6 + len(bucket):]
for obj in s3.list_objects_v2(Bucket=bucket, Prefix=adapter_key)["Contents"]:
    dest = work / "adapter" / obj["Key"].rsplit("/", 1)[-1]
    dest.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(bucket, obj["Key"], str(dest))
with tarfile.open(work / "model.tar.gz", "w:gz") as tar:
    tar.add(work / "adapter", arcname="adapter")
    for name in ("inference.py", "requirements.txt"):
        tar.add(f"serving/{name}", arcname=f"code/{name}")
model_data_uri = model_prefix + "serving/model.tar.gz"
s3.upload_file(str(work / "model.tar.gz"), bucket, model_data_uri[6 + len(bucket):])

base, tuned = results["base"]["exact_match"], results["finetuned"]["exact_match"]
passed = tuned >= MIN_EXACT_MATCH and tuned > base
print(f"gate: fine-tuned exact match {tuned:.0%} (needs >= {MIN_EXACT_MATCH:.0%}), base {base:.0%} -> "
      f"{'PASS' if passed else 'FAIL'}")

try:
    sm.create_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP,
                                  ModelPackageGroupDescription="CPU SFT lab: booking request -> JSON (Qwen3-0.6B + LoRA)")
except sm.exceptions.ClientError as e:
    if "already exists" not in str(e):
        raise
package_arn = sm.create_model_package(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    ModelPackageDescription=f"adapter from {training_job}, evaluated by {eval_job}",
    ModelApprovalStatus="Approved" if passed else "Rejected",
    InferenceSpecification={
        "Containers": [{
            "Image": f"763104351884.dkr.ecr.{region}.amazonaws.com/pytorch-inference:2.6-cpu-py312",
            "ModelDataUrl": model_data_uri,
            "Environment": {"SAGEMAKER_PROGRAM": "inference.py", "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
                            "SAGEMAKER_MODEL_SERVER_WORKERS": "1"},
        }],
        "SupportedContentTypes": ["application/json"],
        "SupportedResponseMIMETypes": ["application/json"],
    },
    ModelMetrics={"ModelQuality": {"Statistics": {"ContentType": "application/json", "S3Uri": eval_uri}}},
    CustomerMetadataProperties={"base_model": results["base_model"], "training_job": training_job,
                                "evaluation_job": eval_job, "base_exact_match": f"{base:.3f}",
                                "finetuned_exact_match": f"{tuned:.3f}"},
)["ModelPackageArn"]
print(f"registered: {package_arn} ({'Approved' if passed else 'Rejected'})")

In SageMaker Studio, **Models** > **Registered models** > `cpu-ft-sft-booking` shows each version with its status,
metadata and evaluation file. A rejected version can still be approved there (or with
`update_model_package`) if you decide to deploy it anyway.

### In MLflow

The training jobs log their parameters and the loss per step; the evaluation job logs the base and fine-tuned
scores, all in the experiment `cpu-fine-tuning-techniques` (one run per job; notebooks 02-04 log there too). The
link is valid for a few minutes; re-run the cell for a new one.

In [ ]:
print(mlflow_url(sm, mlflow_arn) or "MLflow is off")

## 4. Deploy the approved model: three ways

The newest **approved** version in the registry is deployed three times. All three endpoints run the same model
data and differ only in how capacity is provided:

| endpoint | What SageMaker does | Billing |
|---|---|---|
| `realtime` | one `ml.m5.xlarge` instance, always on | per instance-hour |
| `realtime-pools` | the first type in a priority list that has capacity (`InstancePools`, up to 5) | per instance-hour |
| `serverless` | starts compute on demand; the first request after idle time waits for a cold start | per request (compute time) |

At startup each endpoint downloads the base model from the Hugging Face Hub and loads the adapter on top
(`serving/inference.py`). The adapter is kept separate rather than merged, so a request can ask for the base model
(`"model": "base"`) and you can compare both on the same endpoint. Creating the endpoints takes a few minutes.

In [ ]:
approved = sm.list_model_packages(ModelPackageGroupName=MODEL_PACKAGE_GROUP, ModelApprovalStatus="Approved",
                                  SortBy="CreationTime", SortOrder="Descending", MaxResults=1)["ModelPackageSummaryList"]
if not approved:
    raise RuntimeError(f"No approved version in {MODEL_PACKAGE_GROUP}: the model did not pass the gate in step 3. "
                       "Review it in the Model Registry, and approve it there to deploy it anyway.")
package_arn = approved[0]["ModelPackageArn"]
print("deploying", package_arn)

variants = {
    "realtime": {"InstanceType": "ml.m5.xlarge", "InitialInstanceCount": 1},
    "realtime-pools": {"InstancePools": [{"InstanceType": t, "Priority": i + 1}
                                         for i, t in enumerate(["ml.m5.xlarge", "ml.m7i.xlarge", "ml.c7i.2xlarge"])],
                       "InitialInstanceCount": 1, "VariantInstanceProvisionTimeoutInSeconds": 1800},
    "serverless": {"ServerlessConfig": {"MemorySizeInMB": 6144, "MaxConcurrency": 1}},
}
stamp = time.strftime("%Y%m%d-%H%M%S")
endpoints = {}
for kind, variant in variants.items():
    name = f"cpu-ft-sft-{kind}-{stamp}"
    # The model is created from the package: image, model data and environment come from the registry
    sm.create_model(ModelName=name, ExecutionRoleArn=role, PrimaryContainer={"ModelPackageName": package_arn})
    sm.create_endpoint_config(EndpointConfigName=name,
                              ProductionVariants=[{"VariantName": "AllTraffic", "ModelName": name, **variant}])
    sm.create_endpoint(EndpointName=name, EndpointConfigName=name)
    endpoints[kind] = name
t0 = time.time()
for kind, name in endpoints.items():
    try:
        sm.get_waiter("endpoint_in_service").wait(EndpointName=name, WaiterConfig={"Delay": 20, "MaxAttempts": 90})
        print(f"{kind:15} in service after {time.time() - t0:.0f}s")
    except Exception:
        print(f"{kind:15} {sm.describe_endpoint(EndpointName=name).get('FailureReason')}")

### Call the endpoints

In [ ]:
from botocore.config import Config

runtime = boto3.client("sagemaker-runtime", region_name=region,
                       config=Config(read_timeout=120, retries={"max_attempts": 2}))


def invoke(endpoint, request, model="finetuned"):
    resp = runtime.invoke_endpoint(EndpointName=endpoint, ContentType="application/json",
                                   Body=json.dumps({"request": request, "model": model}))
    return json.loads(resp["Body"].read())


endpoints = globals().get("endpoints") or lab_endpoints()  # after a kernel restart
request = "hey! Dana - four ppl friday 7pm if possible"
print(f"request: {request}\n")
for kind, name in endpoints.items():
    for attempt in ("first", "second"):
        t = time.time()
        try:
            out = invoke(name, request)["booking"]
        except Exception as e:
            out = f"error: {str(e)[:120]}"
        print(f"{kind:15} {attempt:6} {time.time() - t:5.1f}s  {out}")

**How to read the output.** Each endpoint is called twice with the same request; the answer should be the same
JSON from all three. The time is the full round trip from this notebook:

* `realtime` and `realtime-pools`: a few seconds both times. The instance is already running with the model
  loaded, and you pay for it per hour whether it is used or not.
* `serverless`: the first call includes a **cold start** (SageMaker starts compute, downloads the container and
  the base model, loads the adapter: tens of seconds); the second call is warm. You pay only while requests are
  processed, so it suits low or irregular traffic that can tolerate an occasional slow first request.

`realtime-pools` behaves like `realtime` here; its value shows when the first instance type has no capacity and
SageMaker moves down the list.

### Base vs fine-tuned, on the same endpoint

Try your own requests: the same endpoint answers with the adapter (`finetuned`) and without it (`base`).

In [ ]:
for request in ["hey! Dana - four ppl friday 7pm if possible",
                "Could I book a table for 2 next Tuesday around 8:30pm? Name is Marco",
                "Lena here, we're 6, saturday lunch at 1pm please"]:
    print(request)
    for model in ("base", "finetuned"):
        out = invoke(endpoints["realtime"], request, model)
        print(f"  {model:10} {out['raw'] if out['booking'] is None else json.dumps(out['booking'])!r}")

## 5. Clean up

The real-time endpoints bill per instance-hour until they are deleted; the serverless one costs nothing while
idle but is deleted too. The registry entries, the S3 data and the MLflow runs cost little or nothing; keep them
to look at later, or set the flags below to delete them as well.

In [ ]:
DELETE_REGISTRY = False  # True: also delete the model package group and its versions
DELETE_S3_DATA = False   # True: also delete everything under s3://{bucket}/{prefix}/ (all four labs)

# Every endpoint step 4 created (also earlier runs, or after a kernel restart); model and config share its name
for name in {**lab_endpoints(), **globals().get("endpoints", {})}.values():
    for delete, kw in ((sm.delete_endpoint, "EndpointName"), (sm.delete_endpoint_config, "EndpointConfigName"),
                       (sm.delete_model, "ModelName")):
        try:
            delete(**{kw: name})
        except sm.exceptions.ClientError as e:
            print(f"  {name}: {e.response['Error']['Message']}")
    print("deleted endpoint, endpoint config and model:", name)
left = [e["EndpointName"] for e in sm.list_endpoints(NameContains="cpu-ft-sft-")["Endpoints"]
        if e["EndpointStatus"] != "Deleting"]
print("other cpu-ft-sft endpoints still in the account:", left or "none")

if DELETE_REGISTRY:
    for p in sm.get_paginator("list_model_packages").paginate(ModelPackageGroupName=MODEL_PACKAGE_GROUP):
        for v in p["ModelPackageSummaryList"]:
            sm.delete_model_package(ModelPackageName=v["ModelPackageArn"])
    sm.delete_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP)
    print("deleted model package group", MODEL_PACKAGE_GROUP)
if DELETE_S3_DATA:
    s3r = boto3.resource("s3", region_name=region)
    s3r.Bucket(bucket).objects.filter(Prefix=f"{prefix}/").delete()
    print(f"deleted s3://{bucket}/{prefix}/")

## What you saw

* **Prepare:** data goes to a job through S3 input channels; results come back to S3 from `/opt/ml/model`.
* **Fine-tune:** a training job produces a small LoRA adapter on top of a public base model. **Capacity** is a
  choice: on-demand for predictability, Spot for cost, a fallback list when a type is scarce.
* **Evaluate:** a separate job scores base and fine-tuned on held-out data; a **quality gate** decides what is
  registered as approved in the **Model Registry**; **MLflow** keeps every run's parameters and metrics.
* **Deploy:** only an approved version is deployed, as base model + adapter, on a real-time endpoint (always on,
  optionally with instance pools as fallback) or a serverless one (pay per request, cold starts).
* **Clean up:** endpoints are what keep billing.

Next: `02-continued-pretraining.ipynb`, `03-dpo.ipynb` and `04-grpo.ipynb` apply the same pattern to other
techniques, more compactly (training and evaluation in one job).